# VLM2Vec V2 — LongDocURL Fine-tuning

**流程：**

1. 挂载 Google Drive（数据在 Drive 里）
2. Clone VLM2Vec 并安装依赖
3. 上传 / 注册自定义 dataset loader
4. 写入配置文件
5. 启动训练

> 确保 Runtime → Change runtime type 已选 **A100 GPU**


## Cell 0 — 检查 GPU


In [2]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

Tue Mar 24 04:59:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Cell 1 — 挂载 Google Drive


In [3]:
from google.colab import drive

drive.mount("/content/drive")

# ── 修改为你的实际路径 ──────────────────────────────────────────
import os

DRIVE_BASE = "/content/drive/MyDrive/vlm2vec_longdocurl"  # Drive 根目录
TRIPLETS_TRAIN = f"{DRIVE_BASE}/data/triplets/triplets_train.jsonl"
TRIPLETS_VAL = f"{DRIVE_BASE}/data/triplets/triplets_val.jsonl"
PAGE_IMAGES_DIR = f"{DRIVE_BASE}/data/triplets/page_images"  # PNG 渲染目录
OUTPUT_DIR = f"{DRIVE_BASE}/outputs/longdocurl_vlm2vec"

# 验证路径
for p in [TRIPLETS_TRAIN, TRIPLETS_VAL, PAGE_IMAGES_DIR]:
    exists = os.path.exists(p)
    print(f'  {"✓" if exists else "✗"} {p}')

os.makedirs(OUTPUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✓ /content/drive/MyDrive/vlm2vec_longdocurl/data/triplets/triplets_train.jsonl
  ✓ /content/drive/MyDrive/vlm2vec_longdocurl/data/triplets/triplets_val.jsonl
  ✓ /content/drive/MyDrive/vlm2vec_longdocurl/data/triplets/page_images


In [3]:
! pip list


Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
absolufy-imports                      0.3.1
accelerate                            1.10.1
aiofiles                              24.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.13.0
aiosignal                             1.4.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
alembic                               1.16.5
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.9.3
anyio                                 4.11.0
anywidget                             0.9.18
argon2-cffi                           25.1.0
argon2-cffi-bindings                  25.1.0
array_record                          0.8.1
arrow 

## Cell 2 — Clone VLM2Vec & 安装依赖


In [4]:
import os
if not os.path.exists('/content/VLM2Vec'):
    !git clone https://github.com/TimothyXia9/VLM2Vec-finetune /content/VLM2Vec
else:
    print('VLM2Vec already cloned, skipping.')

%cd /content/VLM2Vec

VLM2Vec already cloned, skipping.
/content/VLM2Vec


In [5]:

!pip install -r requirements.txt


## Cell 3 — 注册自定义 Dataset Loader


In [6]:
loader_code = """
from __future__ import annotations
import json
import io
from pathlib import Path
from PIL import Image
from datasets import Dataset
from src.data.dataset.base_pair_dataset import (
    AutoPairDataset, add_metainfo_hook, MULTIMODAL_FEATURES, RESOLUTION_MAPPING,
)
from src.model.processor import VLM_IMAGE_TOKENS

QUERY_PROMPT = (
    "This query is related to retrieving a relevant page from a multi-page document. "
    "The retrieved page should contain text, tables, figures, or structured information "
    "necessary to answer the query: "
)
PAGE_PROMPT = "A page from a document containing text, tables, or figures."

def _read_image_bytes(path: str, max_size: int = 384) -> bytes:
    try:
        img = Image.open(path).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        return buf.getvalue()
    except Exception:
        return b""

@add_metainfo_hook
def data_prepare(batch_dict, *args, **kwargs):
    model_backbone   = kwargs["model_backbone"]
    image_resolution = kwargs["image_resolution"]
    image_token = VLM_IMAGE_TOKENS[model_backbone]
    resolution  = RESOLUTION_MAPPING.get(image_resolution, None)

    query_texts, query_images = [], []
    pos_texts,   pos_images   = [], []
    neg_texts,   neg_images   = [], []

    for question, pos_path, neg_paths in zip(
        batch_dict["question"],
        batch_dict["pos_image_path"],
        batch_dict["neg_image_paths"],
    ):
        query_texts.append(f"{QUERY_PROMPT}{question}")
        query_images.append({"bytes": [None], "paths": [None], "resolutions": [[224, 224]]})

        pos_bytes = _read_image_bytes(pos_path) if pos_path and Path(pos_path).exists() else b""
        pos_texts.append(f"{PAGE_PROMPT} {image_token}")
        pos_images.append({"bytes": [pos_bytes], "paths": [pos_path], "resolutions": [resolution or [224, 224]]})

        neg_t, neg_i = [], []
        for neg_path in neg_paths:
            neg_bytes = _read_image_bytes(neg_path) if neg_path and Path(neg_path).exists() else b""
            neg_t.append(f"{PAGE_PROMPT} {image_token}")
            neg_i.append({"bytes": [neg_bytes], "paths": [neg_path], "resolutions": [resolution or [224, 224]]})
        if not neg_t:
            neg_t.append("")
            neg_i.append({"bytes": [b""], "paths": [""], "resolutions": [[224, 224]]})
        neg_texts.append(neg_t)
        neg_images.append(neg_i)

    return {"query_text": query_texts, "query_image": query_images,
            "pos_text": pos_texts, "pos_image": pos_images,
            "neg_text": neg_texts, "neg_image": neg_images}


DATASET_PARSER_NAME = "longdocurl"

@AutoPairDataset.register(DATASET_PARSER_NAME)
def load_longdocurl_dataset(model_args, data_args, training_args, *args, **kwargs):
    dataset_path = kwargs.get("dataset_path")
    if not dataset_path:
        raise ValueError("longdocurl loader requires `dataset_path` in yaml config")
    records = []
    with open(dataset_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            r = json.loads(line)
            neg_paths = [c["image_path"] for c in r.get("hard_neg_chunks", []) if c.get("image_path")]
            records.append({"question": r["question"], "answer": r["answer"],
                            "pos_image_path": r["pos_chunk"]["image_path"],
                            "neg_image_paths": neg_paths})
    num_sample = kwargs.get("num_sample_per_subset", getattr(data_args, "num_sample_per_subset", None))
    if num_sample and num_sample < len(records):
        records = records[:int(num_sample)]
    num_rows = len(records)
    dataset = Dataset.from_list(records)
    num_shards = training_args.dataloader_num_workers if training_args.dataloader_num_workers > 0 else 1
    dataset = dataset.to_iterable_dataset(num_shards=num_shards)
    kwargs["model_backbone"]      = model_args.model_backbone
    kwargs["image_resolution"]    = data_args.image_resolution
    kwargs["global_dataset_name"] = kwargs.get("global_dataset_name", f"longdocurl/{Path(dataset_path).stem}")
    dataset = dataset.map(lambda x: data_prepare(x, **kwargs),
                          batched=True, batch_size=32,
                          remove_columns=["question", "answer", "pos_image_path", "neg_image_paths"],
                          drop_last_batch=True, features=MULTIMODAL_FEATURES)
    setattr(dataset, "num_rows", num_rows)
    return dataset
"""

loader_path = "/content/VLM2Vec/src/data/dataset/longdocurl_dataset.py"
with open(loader_path, "w") as f:
    f.write(loader_code)

# __init__.py 里注册
init_path = "/content/VLM2Vec/src/data/dataset/__init__.py"
init_text = open(init_path).read()
import_line = "from src.data.dataset.longdocurl_dataset import *\n"
if import_line not in init_text:
    with open(init_path, "a") as f:
        f.write(import_line)
    print("✓ Loader registered in __init__.py")
else:
    print("✓ Loader already registered")

✓ Loader already registered


## Cell 4 — 写入配置文件


In [7]:
import os, yaml

EXP_DIR = "/content/VLM2Vec/experiments/longdocurl"
os.makedirs(EXP_DIR, exist_ok=True)

# ── 数据配置 ────────────────────────────────────────────────────
data_config = {
    "longdocurl_train": {
        "dataset_parser": "longdocurl",
        "dataset_path": TRIPLETS_TRAIN,
        "weight": 1,
    }
}
with open(f"{EXP_DIR}/data_config.yaml", "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)
print("✓ data_config.yaml written")

# ── 训练超参 ────────────────────────────────────────────────────
# 可以在这里直接修改超参
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # 换 7B: 'Qwen/Qwen2-VL-7B-Instruct'
LORA_R = 16
LORA_ALPHA = 64
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR = 2e-5
EPOCHS = 3
MAX_PIXELS = 501760  # 28*28*640，减小可节省显存
MAX_STEPS = (1174 // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
print(f"Model:          {MODEL_NAME}")
print(f"LoRA r/alpha:   {LORA_R}/{LORA_ALPHA}")
print(f"Effective batch:{BATCH_SIZE * GRAD_ACCUM}")
print(f"Learning rate:  {LR}")
print(f"Epochs:         {EPOCHS}")
print(f"Output dir:     {OUTPUT_DIR}")

✓ data_config.yaml written
Model:          Qwen/Qwen2-VL-2B-Instruct
LoRA r/alpha:   16/64
Effective batch:16
Learning rate:  2e-05
Epochs:         3
Output dir:     /content/drive/MyDrive/vlm2vec_longdocurl/outputs/longdocurl_vlm2vec


## Cell 5 — 启动训练


In [ ]:
!pip

In [1]:
!pip install --upgrade "wandb>=0.24.1,<0.25"
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tianqi5902 (tianqi5902-new-york-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
import torch, sys
print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)



python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.8.0+cu126
cuda: 12.6


In [9]:
from flash_attn.flash_attn_interface import flash_attn_varlen_qkvpacked_func
print('OK')

OK


In [21]:
%cd /content/VLM2Vec


!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m torch.distributed.run --nproc_per_node 1 train.py \
    --model_name Qwen/Qwen2-VL-2B-Instruct \
    --model_backbone qwen2_vl \
    --pooling last \
    --normalize true \
    --temperature 0.1 \
    --lora true \
    --lora_r 16 \
    --lora_alpha 64 \
    --lora_dropout 0.05 \
    --lora_target_modules "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj" \
    --dataset_config /content/VLM2Vec/experiments/longdocurl/data_config.yaml \
    --gradient_checkpointing true \
    --resize_use_processor true \
    --resize_min_pixels 3136 \
    --resize_max_pixels 50176 \
    --num_hardneg 1 \
    --output_dir /content/drive/MyDrive/vlm2vec_longdocurl/outputs/longdocurl_vlm2vec \
    --max_steps 1 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.05 \
    --lr_scheduler_type cosine \
    --bf16 true \
    --tf32 true \
    --dataloader_num_workers 0 \
    --logging_steps 1 \
    --image_encoder_freeze true \
    --grad_cache true \
    --gc_q_chunk_size 1 \
    --gc_p_chunk_size 1 \
    --interleave_stopping_strategy first_exhausted \
    --remove_unused_columns false \
    --resume_from none \
    --report_to none 2>&1 | grep -E "loss|grad_norm"

/content/VLM2Vec
[2026-03-24 05:13:03,812] INFO [numexpr.utils:164] NumExpr defaulting to 12 threads.
[2026-03-24 05:13:04,071] INFO [datasets:112] TensorFlow version 2.19.0 available.
[2026-03-24 05:13:04,072] INFO [datasets:125] JAX version 0.5.3 available.
2026-03-24 05:13:07.840102: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 05:13:07.858841: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774329187.881095   16244 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774329187.887821   16244 cuda_blas.cc:1407] Unab

In [19]:
model_py = "/content/VLM2Vec/src/model/model.py"
with open(model_py, "r") as f:
    lines = f.readlines()

# 清理所有 DEBUG 行
lines = [line for line in lines if "[DEBUG]" not in line]

# 找 compute_similarity 位置
for i, line in enumerate(lines):
    if "scores = self.compute_similarity" in line:
        insert_pos = i + 1
        print(f"插入位置: 第 {i+1} 行后")
        break

debug_lines = [
    "        print(f'[DEBUG] qry norm: {all_qry_reps.norm(dim=-1).mean().item():.4f} shape: {all_qry_reps.shape}')\n",
    "        print(f'[DEBUG] tgt norm: {all_tgt_reps.norm(dim=-1).mean().item():.4f} shape: {all_tgt_reps.shape}')\n",
    "        print(f'[DEBUG] scores: {scores}')\n",
]

for i, line in enumerate(reversed(debug_lines)):
    lines.insert(insert_pos, line)

with open(model_py, "w") as f:
    f.writelines(lines)

# 验证
for i, line in enumerate(lines[insert_pos-2:insert_pos+6], start=insert_pos-1):
    print(f"{i+1}: {line}", end="")

插入位置: 第 311 行后
311: 
312:         scores = self.compute_similarity(all_qry_reps, all_tgt_reps)
313:         print(f'[DEBUG] qry norm: {all_qry_reps.norm(dim=-1).mean().item():.4f} shape: {all_qry_reps.shape}')
314:         print(f'[DEBUG] tgt norm: {all_tgt_reps.norm(dim=-1).mean().item():.4f} shape: {all_tgt_reps.shape}')
315:         print(f'[DEBUG] scores: {scores}')
316:         scores = scores.view(all_qry_reps.size(0), -1)
317:         target = torch.arange(scores.size(0), device=scores.device, dtype=torch.long)
318:         target = target * (all_qry_reps.size(0) // all_tgt_reps.size(0))


## Cell 6 — 验证 checkpoint 输出


In [ ]:
import os

print("Output dir contents:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

## Cell 7 — 快速推理验证（可选）


In [ ]:
import sys

sys.path.insert(0, "/content/VLM2Vec")

import torch
from PIL import Image
from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN2_VL, VLM_IMAGE_TOKENS
from src.model.vlm_backbone.qwen2_vl.qwen_vl_utils import process_vision_info

# 找最新 checkpoint
import os

ckpts = sorted([d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")])
latest_ckpt = os.path.join(OUTPUT_DIR, ckpts[-1]) if ckpts else OUTPUT_DIR
print("Loading checkpoint:", latest_ckpt)

model_args = ModelArguments(
    model_name=MODEL_NAME,
    checkpoint_path=latest_ckpt,
    pooling="last",
    normalize=True,
    model_backbone="qwen2_vl",
    lora=True,
)
data_args = DataArguments()
processor = load_processor(model_args, data_args)
model = MMEBModel.load(model_args).to("cuda", dtype=torch.bfloat16).eval()
print("Model loaded.")

In [ ]:
import json
import numpy as np

# 从 val 集取几条做快速 Recall@1 验证
val_records = []
with open(TRIPLETS_VAL) as f:
    for line in f:
        val_records.append(json.loads(line.strip()))

sample = val_records[:20]  # 只取前20条
image_token = VLM_IMAGE_TOKENS[QWEN2_VL]
PAGE_PROMPT = "A page from a document containing text, tables, or figures."
QUERY_PROMPT = (
    "This query is related to retrieving a relevant page from a multi-page document. "
    "The retrieved page should contain text, tables, figures, or structured information "
    "necessary to answer the query: "
)


def encode_text(text):
    inputs = processor(text=text, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        return model(qry=inputs)["qry_reps"].cpu().float().numpy()


def encode_image(image_path, text):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(text=f"{image_token} {text}", images=[img], return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        return model(tgt=inputs)["tgt_reps"].cpu().float().numpy()


# 计算 Recall@1
hits = 0
for i, r in enumerate(sample):
    q_emb = encode_text(f"{QUERY_PROMPT}{r['question']}")
    pos_emb = encode_image(r["pos_chunk"]["image_path"], PAGE_PROMPT)
    neg_embs = [
        encode_image(c["image_path"], PAGE_PROMPT)
        for c in r["hard_neg_chunks"]
        if c.get("image_path")
    ]

    all_embs = np.vstack([pos_emb] + neg_embs)  # pos at index 0
    scores = (q_emb @ all_embs.T).squeeze()
    if scores.argmax() == 0:
        hits += 1

print(
    f"Recall@1 on {len(sample)} val samples: {hits}/{len(sample)} = {hits/len(sample):.2%}"
)